# CSE428 Project — Pet Segmentation & Breed Classification

**Dataset:** [Oxford-IIIT Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/) · 37 breeds · 3,680 trainval / 3,666 test images

**Models:** U-Net and Attention U-Net with a breed-classifier head on the shared encoder (trained jointly)

**Contents:** data exploration → base U-Net → Attention U-Net → results (mIoU, Dice, pixel accuracy · accuracy, precision, recall, F1 on train/val/test)

> Training runs in **resumable checkpoint chunks**: each notebook version saves its state to the output, and the next run (or a groupmate) resumes from it — never start from scratch.

## 0. Repository sync — code comes from GitHub, no copy-paste

All project code lives in a public repo. This cell clones it (fresh session) or pulls the latest version (existing session).

In [ ]:
import os, sys

REPO_URL = "https://github.com/shahriar-abid/cse428-pets.git"
REPO_DIR = "/kaggle/working/cse428-pets"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("repo ready:", os.getcwd())

## 1. Setup

In [ ]:
import yaml
import torch

from src.utils import seed_everything, get_device, check_device, resolve_output_dir

CFG = yaml.safe_load(open("configs/config.yaml"))
seed_everything(CFG["seed"])
DEVICE = check_device(get_device())
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(DEVICE))
print("device:", DEVICE)
print("config:", CFG)

## 2. Dataset & Exploration

**Oxford-IIIT Pet** — 3,680 trainval + 3,666 test images across 37 breeds.

Each image ships with a **trimap**: `1` = foreground (pet), `2` = background, `3` = boundary (not classified). Per the project guidelines, boundary pixels are merged into the foreground, giving a binary mask: `background → 0`, `foreground + boundary → 1`.

Split: 90% train / 10% validation from trainval (deterministic, seeded — identical across sessions so checkpointed training resumes on the same data). The official test set is used for testing.

In [ ]:
from src.data import get_loaders

DATASETS, LOADERS = get_loaders(
    root=CFG["data"]["root"],
    img_size=CFG["data"]["img_size"],
    val_frac=CFG["data"]["val_frac"],
    seed=CFG["seed"],
    augment=CFG["data"]["augment"],
    batch_size=CFG["data"]["batch_size"],
    num_workers=CFG["data"]["num_workers"],
    download=True,
)
for name, d in DATASETS.items():
    print(f"{name}: {len(d)} samples ({len(d.classes)} classes)")

### 2.1 Images with mask overlays (3x3, required format)

In [ ]:
import matplotlib.pyplot as plt
from src.viz import plot_overlay_grid

fig = plot_overlay_grid(DATASETS["val"], figsize=(12, 12))
plt.show()

## 3. Base U-Net — segmentation + classification (trained jointly)

**Architecture** (Ronneberger et al., 2015):
- **Segmentation head** on the decoder: 1×1 conv → binary mask, trained with BCE + Dice loss
- **Classifier head** on the encoder bottleneck: global average pooling → linear → 37 breeds, cross-entropy
- **Joint loss**: `L = L_seg + λ · L_cls` (both heads train together, sharing the encoder)

**Checkpoint-chunked training:** each run trains *from the last completed epoch* up to `train.epochs_total`, saving `checkpoints/last.pth` (full state) and `checkpoints/best.pth` (best val mIoU) to the notebook output.

**To continue training later (or as a groupmate):** *Add Input → this notebook's previous version output* — the trainer finds the checkpoint automatically. Just bump `epochs_total` and run.

In [ ]:
from src.models import build_model
from src.train import Trainer, find_resume_checkpoint

MODEL_NAME = "unet"                  # "unet" | "attention_unet"
CFG["model"]["name"] = MODEL_NAME
CFG["train"]["epochs_total"] = 15    # bump each version: 15 -> 30 -> 45 -> 60

OUT_DIR = os.path.join(resolve_output_dir(CFG["output"]["dir"]), MODEL_NAME)
os.makedirs(OUT_DIR, exist_ok=True)
print("output dir:", OUT_DIR)
model = build_model(CFG).to(DEVICE)
resume_ckpt = find_resume_checkpoint(out_dir=OUT_DIR, model_name=MODEL_NAME)
print("resume checkpoint:", resume_ckpt)
trainer = Trainer(model, LOADERS, DEVICE, CFG, out_dir=OUT_DIR, resume=resume_ckpt)

In [ ]:
history = trainer.fit()

### 3.1 Training curves — loss + metric update per epoch (requirement)

In [ ]:
from src.viz import plot_history

fig = plot_history(trainer.history)
plt.show()

### 3.2 Result summary — train / validation / test (requirement)

Segmentation: **mIoU, Dice coefficient, pixel accuracy** · Classification: **accuracy, precision, recall, F1** (macro-averaged over the 37 breeds).

In [ ]:
import pandas as pd

results = trainer.final_report()
rows = [{"split": split, **m["seg"], **m["cls"]} for split, m in results.items()]
pd.DataFrame(rows).set_index("split").round(4)

### 3.2b Persisted outputs (for resume / groupmate handoff)

Verify the checkpoint + report files are on disk inside `/kaggle/working` so Kaggle captures them into this version's output — the next chunk (or a groupmate) attaches this version and resumes from `best.pth`/`last.pth`.

In [ ]:
import glob

for p in sorted(glob.glob(os.path.join(OUT_DIR, "**", "*.*"), recursive=True)):
    print(f"{os.path.getsize(p):>12,} bytes  {p}")

### 3.3 Predictions — image | ground truth | model output (required format)

Random validation samples; titles show the predicted breed with confidence and the per-image foreground IoU.

In [ ]:
from src.viz import plot_prediction_grid

fig = plot_prediction_grid(DATASETS["val"], trainer.model, DEVICE, nrows=3, figsize=(13, 13))
plt.show()

## 4. Attention U-Net — segmentation + classification

*(same protocol as section 3 with `MODEL_NAME = "attention_unet"` — cells land on day 4; see Oktay et al., 2018: additive attention gates re-weight each skip connection using the coarser decoder feature as the gating signal)*

## 5. U-Net vs Attention U-Net — comparison & discussion

*(filled after both models finish training)*